<font color="#CA0032"><h1>Practica B3-T5 · Laboratorio de estrategias</h1></font>

## Comparativa de arquitecturas para prediccion de ventas Rossmann

**Grupo:** Alonso Diaz · Raul Rodriguez · Piettro Enrico

---

Este notebook recoge **tres estrategias** para predecir las ventas de las 9 tiendas objetivo,
tal como sugiere el enunciado. El objetivo es comparar sus R2 y justificar la eleccion final.

| Seccion | Responsable | Estrategia |
|---------|-------------|------------|
| A | Alonso | **Un modelo por tienda** (9 modelos LSTM pequenos) |
| B | Raul | **Un modelo por grupo** (2 modelos: cierran-domingos vs nunca-cierran) |
| C | Piettro | **Modelo global** con embedding de `Store` (1 modelo para todas) |

**Base comun:** mismas tiendas, mismo split, mismas metricas -> los R2 son comparables directamente.

## 0. Configuracion comun

> **Todos los miembros usan exactamente esta celda sin modificar.**

In [ ]:
COLAB = False          # True en Google Colab (descarga datos automaticamente)
RUTA_DATA = 'data'     # ruta local; en Colab se sobreescribe abajo

# Tiendas objetivo del enunciado
TIENDAS        = [1, 2, 3, 4, 5, 562, 682, 733, 769]
TIENDAS_CIERRAN = [1, 2, 3, 4, 5]          # cierran los domingos
TIENDAS_ABIERTAS = [562, 682, 733, 769]    # nunca cierran

# Split temporal (enunciado)
F_TEST = '2015-01-01'   # test = desde esta fecha (inclusive)
F_VAL  = '2014-11-15'   # validacion = desde aqui hasta F_TEST

W    = 28               # ventana lookback (dias)
SEED = 7

## 1. Imports

In [ ]:
import numpy as np, pandas as pd, zipfile, os, warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, GRU,
                                     Embedding, Flatten, concatenate, Dropout)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__)

## 2. Carga de datos

In [ ]:
if COLAB:
    import subprocess
    subprocess.run(['git','clone',
        'https://github.com/alonsodt/B3_T5-NN-con-entrdas-hetereogeneas-.git'], check=True)
    RUTA_DATA = 'B3_T5-NN-con-entrdas-hetereogeneas-/data'

with zipfile.ZipFile(os.path.join(RUTA_DATA, 'train.zip')) as z:
    train = pd.read_csv(z.open('train.csv'), parse_dates=['Date'], low_memory=False)
store = pd.read_csv(os.path.join(RUTA_DATA, 'store.csv'))

df = (train.merge(store, on='Store', how='left')
           .sort_values(['Store','Date'])
           .reset_index(drop=True))

# Feature engineering basico
df['StateHoliday'] = df['StateHoliday'].astype(str).replace('0','n')
df['Month']        = df.Date.dt.month
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())
df['CompDistLog']  = np.log1p(df['CompetitionDistance'])
df['Promo2']       = df['Promo2'].fillna(0).astype(int)
df['y']            = np.log1p(df['Sales'])   # objetivo en escala log

# Restringimos a las 9 tiendas del enunciado
df = df[df.Store.isin(TIENDAS)].reset_index(drop=True)
print('Filas:', len(df), '| Tiendas:', df.Store.nunique())

## 3. EDA rapido

### 3.1 Resumen por tienda y metadatos

Estadisticos descriptivos de ventas y atributos estaticos (`StoreType`, `Assortment`, distancia a la
competencia). Sirve para detectar tiendas de escala/comportamiento distinto.

In [ ]:
resumen = []
for sid in TIENDAS:
    g  = df[df.Store == sid]
    go = g[g.Open == 1]
    resumen.append({
        'Store': sid,
        'Familia': 'cierra_dom' if sid in TIENDAS_CIERRAN else 'abre_siempre',
        'StoreType': g['StoreType'].iloc[0],
        'Assortment': g['Assortment'].iloc[0],
        'CompDist_m': int(g['CompetitionDistance'].iloc[0]),
        'dias': len(g),
        '%_abierto': round(100*g['Open'].mean(), 1),
        'venta_media': int(go['Sales'].mean()),
        'venta_mediana': int(go['Sales'].median()),
        'venta_std': int(go['Sales'].std()),
        'venta_max': int(go['Sales'].max()),
        'promo_%': round(100*g['Promo'].mean(), 1),
    })
resumen = pd.DataFrame(resumen).set_index('Store')
print('Rango de fechas:', df.Date.min().date(), '->', df.Date.max().date())
resumen

### 3.2 Series temporales y las dos familias

El profesor distingue dos perfiles: tiendas de barrio que **cierran domingos** (muy periodicas) y tiendas
que **abren siempre** (mas irregulares). Visualizamos la serie completa y un zoom de 60 dias para ver el
ciclo semanal.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 8))
for ax, sid in zip(axes.flat, TIENDAS):
    g = df[df.Store == sid]
    color = 'tab:blue' if sid in TIENDAS_CIERRAN else 'tab:red'
    ax.plot(g.Date, g.Sales, lw=0.5, color=color)
    fam = 'cierra dom' if sid in TIENDAS_CIERRAN else 'abre siempre'
    ax.set_title(f'Tienda {sid} ({fam})', fontsize=9)
    ax.tick_params(axis='x', rotation=45, labelsize=6)
plt.suptitle('Serie completa  |  azul=cierra domingos, rojo=abre siempre', y=1.01)
plt.tight_layout(); plt.show()

# Zoom 60 dias: una tienda de cada familia
fig, axes = plt.subplots(1, 2, figsize=(15, 3))
for ax, sid in zip(axes, [1, 562]):
    g = df[df.Store == sid].iloc[:60]
    ax.plot(g.Date, g.Sales, 'o-', ms=3)
    ax.set_title(f'Tienda {sid} - primeros 60 dias (ciclo semanal)')
    ax.tick_params(axis='x', rotation=45, labelsize=7); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

### 3.3 Distribucion de ventas y transformacion logaritmica

Las ventas presentan **asimetria a la derecha** (cola de dias de alto volumen). Aplicar `log1p(Sales)`
reduce esa asimetria y estabiliza el entrenamiento (la red parte de pesos pequenos y predice mejor
magnitudes acotadas). **Decision:** el objetivo del modelo es `y = log1p(Sales)`; al evaluar
des-transformamos con `expm1`. El valor de asimetria se imprime en la propia figura.

In [ ]:
ventas = df[df.Open == 1]['Sales']
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].hist(ventas, bins=60, color='tab:orange')
ax[0].set_title(f'Sales (asimetria={ventas.skew():.2f})'); ax[0].set_xlabel('Sales')
ax[1].hist(np.log1p(ventas), bins=60, color='tab:green')
ax[1].set_title(f'log1p(Sales) (asimetria={np.log1p(ventas).skew():.2f})'); ax[1].set_xlabel('log1p(Sales)')
plt.tight_layout(); plt.show()

### 3.4 Estacionalidad y efecto de promociones/festivos

Cuatro vistas en un panel:
- **Dia de la semana**: revela el ciclo semanal (y el cierre dominical de la familia 1).
- **Mes**: ciclo anual (caidas en verano, picos en diciembre).
- **Promo**: cuanto sube la venta media con promocion activa.
- **Festivo escolar**: efecto de `SchoolHoliday`.

**Decision:** estas variables de calendario se conocen de antemano -> entran como exogenas/categoricas
(embeddings de `DayOfWeek` y `Month`); `Promo` y `SchoolHoliday` ademas viajan dentro de la ventana LSTM.

In [ ]:
op = df[df.Open == 1]
fig, ax = plt.subplots(2, 2, figsize=(14, 7))

# ventas por dia de semana, por familia
for fam, tiendas, col in [('cierra dom', TIENDAS_CIERRAN, 'tab:blue'),
                          ('abre siempre', TIENDAS_ABIERTAS, 'tab:red')]:
    sub = df[df.Store.isin(tiendas)].groupby('DayOfWeek')['Sales'].mean()
    ax[0,0].plot(sub.index, sub.values, 'o-', label=fam, color=col)
ax[0,0].set_title('Venta media por dia de semana (1=Lun..7=Dom)')
ax[0,0].set_xlabel('DayOfWeek'); ax[0,0].legend(); ax[0,0].grid(alpha=.3)

# ventas por mes
mes = op.groupby('Month')['Sales'].mean()
ax[0,1].bar(mes.index, mes.values, color='tab:purple')
ax[0,1].set_title('Venta media por mes (ciclo anual)'); ax[0,1].set_xlabel('Mes')

# efecto promo
pr = op.groupby('Promo')['Sales'].mean()
ax[1,0].bar(['Sin promo','Con promo'], pr.values, color=['gray','tab:green'])
inc = 100*(pr.get(1,0)-pr.get(0,0))/pr.get(0,1)
ax[1,0].set_title(f'Efecto Promo (+{inc:.0f}% de media)')

# efecto festivo escolar
sh = op.groupby('SchoolHoliday')['Sales'].mean()
ax[1,1].bar(['Normal','Festivo escolar'], [sh.get(0,0), sh.get(1,0)], color=['gray','tab:orange'])
ax[1,1].set_title('Efecto SchoolHoliday')
plt.tight_layout(); plt.show()

### 3.5 Autocorrelacion, heterocedasticidad y la trampa de `Customers`

- **Autocorrelacion (ACF):** picos marcados en multiplos de 7 confirman la estacionalidad semanal y
  justifican los lags `lag_7`, `lag_14` y una ventana `W=28` (cubre 4 semanas).
- **Heterocedasticidad:** la desviacion movil de las ventas no es constante (varia con el nivel), tal como
  comento el profesor; el `log` la atenua.
- **`Customers`:** esta casi perfectamente correlacionada con `Sales`. Usarla seria hacer trampa
  (no se conoce de antemano), por eso el enunciado la **prohibe** como entrada.

In [ ]:
def acf(serie, nlags=35):
    x = np.asarray(serie, dtype=float)
    return [1.0] + [np.corrcoef(x[k:], x[:-k])[0,1] for k in range(1, nlags+1)]

fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))

# ACF para una tienda de cada familia
for sid, col in [(1,'tab:blue'), (562,'tab:red')]:
    serie = df[df.Store == sid].sort_values('Date')['Sales'].values
    a = acf(serie, 35)
    ax[0].plot(range(36), a, 'o-', ms=3, label=f'tienda {sid}', color=col)
for k in [7,14,21,28]:
    ax[0].axvline(k, color='k', ls=':', alpha=.3)
ax[0].set_title('ACF (lineas: lag 7,14,21,28)'); ax[0].set_xlabel('lag (dias)')
ax[0].legend(); ax[0].grid(alpha=.3)

# heterocedasticidad: desviacion movil 30 dias
g = df[df.Store == 562].sort_values('Date')
roll = g['Sales'].rolling(30).std()
ax[1].plot(g['Date'], roll, color='tab:red')
ax[1].set_title('Desv. movil 30d (tienda 562) - varianza no constante')
ax[1].tick_params(axis='x', rotation=45, labelsize=7); ax[1].grid(alpha=.3)

# correlacion Customers vs Sales
corr = df[df.Open==1][['Customers','Sales']].corr().iloc[0,1]
ax[2].scatter(df[df.Open==1]['Customers'], df[df.Open==1]['Sales'], s=2, alpha=.2)
ax[2].set_title(f'Customers vs Sales (corr={corr:.2f}) -> PROHIBIDA')
ax[2].set_xlabel('Customers'); ax[2].set_ylabel('Sales')
plt.tight_layout(); plt.show()

### 3.6 Sintesis: decisiones de modelado derivadas del EDA

| Hallazgo del EDA | Decision de modelado |
|---|---|
| Ventas muy asimetricas | Objetivo `log1p(Sales)`, des-transformar con `expm1` |
| Fuerte ciclo semanal (ACF en lag 7,14) | Lags `lag_7`/`lag_14` + ventana `W=28` |
| Ciclo anual (verano/diciembre) | Embedding de `Month` |
| Cierre dominical en familia 1 | Entrenar solo dias abiertos; `Open` en la secuencia |
| Promo sube mucho la venta | `Promo` como exogena dentro de la ventana |
| Heterocedasticidad | El `log` estabiliza la varianza |
| `Customers` ~ `Sales` | Excluida (prohibida por el enunciado) |
| Dos familias de tiendas | Justifica la Estrategia B (un modelo por grupo) |

## 4. Split temporal y enventanado

In [ ]:
CAT_COLS  = ['Store','StoreType','Assortment','StateHoliday','DayOfWeek','Month']
CAT_IDX   = [c+'_idx' for c in CAT_COLS]
NUM_NAMES = ['Promo','SchoolHoliday','Promo2','CompDistLog',
             'lag7','lag14','mean_w','std_w','promo_w']

# Codificacion categoricas
cat_maps, cat_card = {}, {}
for col in CAT_COLS:
    cats = sorted(df[col].unique())
    cat_maps[col] = {v:i for i,v in enumerate(cats)}
    df[col+'_idx'] = df[col].map(cat_maps[col]).astype(int)
    cat_card[col]  = len(cats)

def construir_ventanas(data):
    seq, ynum, y_out, fecha, store_id, open_t = [], [], [], [], [], []
    cats = {c:[] for c in CAT_IDX}
    for _, g in data.groupby('Store'):
        g  = g.sort_values('Date')
        yv = g['y'].values; pr = g['Promo'].values
        sh = g['SchoolHoliday'].values; op = g['Open'].values
        cd = g['CompDistLog'].values;  p2 = g['Promo2'].values
        cv = {c: g[c].values for c in CAT_IDX}
        for i in range(W, len(g)):
            wy = yv[i-W:i]
            seq.append(np.stack([wy, pr[i-W:i], sh[i-W:i], op[i-W:i]], axis=1))
            ynum.append([pr[i], sh[i], p2[i], cd[i],
                         yv[i-7], yv[i-14], wy.mean(), wy.std(), pr[i-W:i].sum()])
            y_out.append(yv[i]); fecha.append(g['Date'].values[i])
            store_id.append(g['Store'].values[i]); open_t.append(op[i])
            for c in CAT_IDX: cats[c].append(cv[c][i])
    out = {'seq': np.array(seq,'float32'), 'num': np.array(ynum,'float32'),
           'y':   np.array(y_out,'float32'), 'fecha': np.array(fecha),
           'store': np.array(store_id), 'open': np.array(open_t)}
    for c in CAT_IDX: out[c] = np.array(cats[c],'int32')
    return out

ds = construir_ventanas(df)
mask = ds['open'] == 1
for k in list(ds): ds[k] = ds[k][mask]

fechas = ds['fecha']
i_tr = np.where(fechas <  np.datetime64(F_VAL))[0]
i_va = np.where((fechas >= np.datetime64(F_VAL)) & (fechas < np.datetime64(F_TEST)))[0]
i_te = np.where(fechas >= np.datetime64(F_TEST))[0]
print(f'train {len(i_tr)} | val {len(i_va)} | test {len(i_te)}')

esc_seq = StandardScaler().fit(ds['seq'][i_tr][:,:,0].reshape(-1,1))
esc_num = StandardScaler().fit(ds['num'][i_tr])

def escala_seq(s):
    s = s.copy()
    sh = s[:,:,0].shape
    s[:,:,0] = esc_seq.transform(s[:,:,0].reshape(-1,1)).reshape(sh)
    return s.astype('float32')

seq_s = escala_seq(ds['seq'])
num_s = esc_num.transform(ds['num']).astype('float32')

def inputs(idx, filtro_store=None):
    if filtro_store is not None:
        mask_s = np.isin(ds['store'][idx], filtro_store)
        idx = idx[mask_s]
    X = {'seq': seq_s[idx], 'num': num_s[idx]}
    for c in CAT_IDX: X[c] = ds[c][idx]
    return X, ds['y'][idx]

ventas_te  = np.expm1(ds['y'][i_te])
store_te   = ds['store'][i_te]

## 5. Metricas y baselines

In [ ]:
def rmspe(real, pred):
    m = real > 0
    return np.sqrt(np.mean(((real[m]-pred[m])/real[m])**2))

def metricas(real, pred, stores, label=''):
    res = {'R2_global':     round(r2_score(real, pred), 4),
           'RMSPE_global':  round(rmspe(real, pred), 4)}
    for sid in TIENDAS:
        m = stores == sid
        if m.sum() > 0:
            res[f'R2_t{sid}'] = round(r2_score(real[m], pred[m]), 4)
    if label: print(label, res)
    return res

# Baseline 1: persistente a 7 dias
pred_p7 = np.expm1(ds['y'][i_te - 7]) if (i_te - 7).min() >= 0 else None

# Baseline 2: media por (tienda, dia de semana)
inv_dow = {v:k for k,v in cat_maps['DayOfWeek'].items()}
df_tr = df[df.Date < pd.Timestamp(F_TEST)]
media_td = df_tr[df_tr.Open==1].groupby(['Store','DayOfWeek'])['Sales'].mean()
dow_te = np.array([inv_dow[i] for i in ds['DayOfWeek_idx'][i_te]])
pred_base = np.array([media_td.get((s,d), df_tr['Sales'].mean())
                      for s,d in zip(store_te, dow_te)])

resultados = {}
resultados['Baseline (media tienda+dia)'] = metricas(ventas_te, pred_base, store_te,
                                                       'Baseline (media tienda+dia)')
print()
print('R2 por tienda:')
for sid in TIENDAS:
    m = store_te == sid
    if m.sum():
        print(f'  Tienda {sid}: {r2_score(ventas_te[m], pred_base[m]):.3f}')

---
## ESTRATEGIA A — Un modelo por tienda
**Responsable: Alonso**

Entrenamos **9 modelos independientes**, uno por tienda. Cada modelo es una LSTM pequena
especializada en la dinamica de esa tienda. Es el punto de partida que recomienda el profesor.

**Ventaja:** muy especializado para cada tienda.
**Desventaja:** poco dato por modelo (~600 muestras); no transfiere conocimiento entre tiendas.

In [ ]:
# TODO (Alonso): implementar y rellenar esta seccion

def modelo_por_tienda(card_store=1):
    """LSTM ligera para una sola tienda (sin embedding de Store)."""
    seq_in  = Input(shape=(W, 4), name='seq')
    h       = LSTM(32, return_sequences=False)(seq_in)
    num_in  = Input(shape=(len(NUM_NAMES),), name='num')
    num_h   = Dense(16, activation='relu')(num_in)
    # Embeddings de las categoricas (excepto Store, que ya no aplica)
    ramas = [h, num_h]
    cat_inputs = []
    for ccol in CAT_IDX:
        if ccol == 'Store_idx': continue   # no embedding de tienda aqui
        card = cat_card[ccol.replace('_idx','')]
        d    = max(2, card // 2)
        inp  = Input(shape=(1,), name=ccol)
        cat_inputs.append(inp)
        ramas.append(Flatten()(Embedding(card, d)(inp)))
    x   = concatenate(ramas)
    x   = Dense(64, activation='relu')(x)
    x   = Dropout(0.3)(x)
    out = Dense(1)(x)
    m   = Model(inputs=[seq_in] + cat_inputs + [num_in], outputs=out)
    m.compile(loss='mse', optimizer='adam')
    return m

def inputs_tienda(idx, sid):
    """Inputs filtrados para una sola tienda (sin 'Store_idx')."""
    mask_s = ds['store'][idx] == sid
    idx_s  = idx[mask_s]
    X = {'seq': seq_s[idx_s], 'num': num_s[idx_s]}
    for c in CAT_IDX:
        if c == 'Store_idx': continue
        X[c] = ds[c][idx_s]
    return X, ds['y'][idx_s]

# --- Entrenamiento (un modelo por tienda) ---
modelos_A = {}
for sid in TIENDAS:
    X_tr, y_tr_s = inputs_tienda(i_tr, sid)
    X_va, y_va_s = inputs_tienda(i_va, sid)
    if len(y_tr_s) == 0:
        print(f'Tienda {sid}: sin datos de entrenamiento, saltando.')
        continue
    m = modelo_por_tienda()
    cbs = [EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)]
    m.fit(X_tr, y_tr_s, validation_data=(X_va, y_va_s),
          epochs=60, batch_size=64, verbose=0, callbacks=cbs)
    modelos_A[sid] = m
    X_te_s, y_te_s = inputs_tienda(i_te, sid)
    pred_s = np.clip(np.expm1(m.predict(X_te_s, verbose=0).flatten()), 0, None)
    real_s = np.expm1(y_te_s)
    r2_s   = r2_score(real_s, pred_s) if len(real_s) > 1 else float('nan')
    print(f'Tienda {sid}: R2={r2_s:.4f}  (n_test={len(real_s)})')

# --- Agregacion de predicciones para metricas globales ---
preds_A = np.zeros(len(i_te))
for sid in TIENDAS:
    if sid not in modelos_A: continue
    mask_s = store_te == sid
    idx_s  = i_te[mask_s]
    X_te_s, _ = inputs_tienda(i_te, sid)
    p = np.clip(np.expm1(modelos_A[sid].predict(X_te_s, verbose=0).flatten()), 0, None)
    preds_A[mask_s] = p

resultados['Estrategia A (por tienda)'] = metricas(ventas_te, preds_A, store_te,
                                                     'Estrategia A (por tienda)')

---
## ESTRATEGIA B — Un modelo por grupo
**Responsable: Raul**

Dividimos las 9 tiendas en **2 grupos** segun su patron de apertura, que el profesor ya
senala como una diferencia importante:

- **Grupo 1** `[1,2,3,4,5]`: tiendas de barrio, **cierran domingos**. Series periodicas y estables.
- **Grupo 2** `[562,682,733,769]`: tipo centro comercial/aeropuerto, **abren todos los dias**. Mas irregulares.

Cada grupo tiene un modelo propio con embedding de `Store` para distinguir tiendas dentro del grupo.

In [ ]:
# TODO (Raul): implementar y rellenar esta seccion

def modelo_grupo(n_stores, dim_emb=4):
    """Modelo para un grupo de tiendas con embedding de Store."""
    seq_in = Input(shape=(W, 4), name='seq')
    h      = LSTM(32, return_sequences=True)(seq_in)
    h      = LSTM(16)(h)
    ramas  = [h]
    cat_inputs = []
    for ccol in CAT_IDX:
        card = cat_card[ccol.replace('_idx','')]
        if ccol == 'Store_idx':
            card = n_stores   # solo las tiendas del grupo
            d    = dim_emb
        else:
            d = max(2, card // 2)
        inp = Input(shape=(1,), name=ccol)
        cat_inputs.append(inp)
        ramas.append(Flatten()(Embedding(card, d)(inp)))
    num_in = Input(shape=(len(NUM_NAMES),), name='num')
    ramas.append(Dense(16, activation='relu')(num_in))
    x   = concatenate(ramas)
    x   = Dense(64, activation='relu')(x)
    x   = Dropout(0.3)(x)
    out = Dense(1)(x)
    m   = Model(inputs=[seq_in] + cat_inputs + [num_in], outputs=out)
    m.compile(loss='mse', optimizer='adam')
    return m

# Re-codificar Store dentro de cada grupo (indices 0..n-1)
def recodificar_store(df_g, tiendas_grupo):
    mapa = {sid: i for i, sid in enumerate(sorted(tiendas_grupo))}
    return df_g['Store'].map(mapa)

grupos = {'G1_cierran': TIENDAS_CIERRAN, 'G2_abiertas': TIENDAS_ABIERTAS}
modelos_B = {}
preds_B   = np.zeros(len(i_te))

for gname, tiendas_g in grupos.items():
    # Recodificamos store_idx para este grupo
    store_map_g = {sid: i for i, sid in enumerate(sorted(tiendas_g))}
    n_g         = len(tiendas_g)

    def inputs_grupo(idx_set, filtro):
        mask_g  = np.isin(ds['store'][idx_set], filtro)
        idx_g   = idx_set[mask_g]
        X = {'seq': seq_s[idx_g], 'num': num_s[idx_g]}
        for c in CAT_IDX:
            if c == 'Store_idx':
                X[c] = np.array([store_map_g[s] for s in ds['store'][idx_g]], dtype='int32')
            else:
                X[c] = ds[c][idx_g]
        return X, ds['y'][idx_g]

    X_tr_g, y_tr_g = inputs_grupo(i_tr, tiendas_g)
    X_va_g, y_va_g = inputs_grupo(i_va, tiendas_g)

    m = modelo_grupo(n_g)
    cbs = [EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)]
    hist = m.fit(X_tr_g, y_tr_g, validation_data=(X_va_g, y_va_g),
                 epochs=60, batch_size=128, verbose=0, callbacks=cbs)
    modelos_B[gname] = m

    X_te_g, y_te_g = inputs_grupo(i_te, tiendas_g)
    p = np.clip(np.expm1(m.predict(X_te_g, verbose=0).flatten()), 0, None)
    mask_g = np.isin(store_te, tiendas_g)
    preds_B[mask_g] = p
    print(f'{gname}: R2={r2_score(np.expm1(y_te_g), p):.4f}')

resultados['Estrategia B (por grupo)'] = metricas(ventas_te, preds_B, store_te,
                                                    'Estrategia B (por grupo)')

---
## ESTRATEGIA C — Modelo global con embedding de Store
**Responsable: Piettro**

Un **unico modelo** que aprende a predecir todas las tiendas simultaneamente. La tienda se
codifica como una entrada categorica con su propio `Embedding`. Es la estrategia que uso el
ganador de la competicion Kaggle de Rossmann y la que el profesor menciona como opcion avanzada.

**Ventaja:** el embedding de Store aprende similitudes entre tiendas; con `ENTRENAR_CON_TODAS=True`
(en Colab/GPU) puede usar las 1.115 tiendas para mejorar la generalizacion.
**Desventaja:** mas complejo; con pocas tiendas puede no superar al modelo por tienda.

In [ ]:
# TODO (Piettro): ajustar hiperparametros y ejecutar en Colab con GPU para mejores resultados

def construir_modelo_global(u1=64, u2=32, dropout=0.3, dim_emb=10, dense=128):
    seq_in = Input(shape=(W, 4), name='seq')
    h      = LSTM(u1, return_sequences=True)(seq_in)
    h      = LSTM(u2)(h)
    ramas  = [h]
    cat_inputs = []
    for ccol in CAT_IDX:
        card = cat_card[ccol.replace('_idx','')]
        d    = int(min(dim_emb, max(2, card // 2)))
        inp  = Input(shape=(1,), name=ccol)
        cat_inputs.append(inp)
        ramas.append(Flatten()(Embedding(card, d)(inp)))
    num_in = Input(shape=(len(NUM_NAMES),), name='num')
    ramas.append(Dense(32, activation='relu')(num_in))
    x   = concatenate(ramas)
    x   = Dense(dense, activation='relu')(x)
    x   = Dropout(dropout)(x)
    x   = Dense(dense // 2, activation='relu')(x)
    out = Dense(1)(x)
    m   = Model(inputs=[seq_in] + cat_inputs + [num_in], outputs=out)
    m.compile(loss='mse', optimizer='adam')
    return m

# Barrido de configuraciones
configs_C = [
    dict(u1=32,  u2=16, dropout=0.3, dim_emb=6,  dense=64),
    dict(u1=64,  u2=32, dropout=0.3, dim_emb=10, dense=128),
    dict(u1=96,  u2=48, dropout=0.4, dim_emb=16, dense=256),
]

resultados_C = []
mejor_C, mejor_r2_C = None, -np.inf

X_tr_C, y_tr_C = inputs(i_tr)
X_va_C, y_va_C = inputs(i_va)
X_te_C, y_te_C = inputs(i_te)

for cfg in configs_C:
    tf.random.set_seed(SEED)
    m = construir_modelo_global(**cfg)
    cbs = [EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]
    m.fit(X_tr_C, y_tr_C, validation_data=(X_va_C, y_va_C),
          epochs=50, batch_size=256, verbose=0, callbacks=cbs)
    pred_c = np.clip(np.expm1(m.predict(X_te_C, verbose=0).flatten()), 0, None)
    met_c  = metricas(ventas_te, pred_c, store_te)
    resultados_C.append({**cfg, **{k: round(v,4) for k,v in met_c.items()}})
    print(cfg, '->', met_c)
    if met_c['R2_global'] > mejor_r2_C:
        mejor_C, mejor_r2_C = m, met_c['R2_global']

pd.DataFrame(resultados_C)

In [ ]:
preds_C = np.clip(np.expm1(mejor_C.predict(X_te_C, verbose=0).flatten()), 0, None)
resultados['Estrategia C (global+embedding)'] = metricas(ventas_te, preds_C, store_te,
                                                          'Estrategia C (global+embedding)')

# Visualizacion del embedding del dia de la semana
emb_layers = {l.name: l for l in mejor_C.layers if 'embedding' in l.name}
key_dow = [k for k in emb_layers if 'dayofweek' in k.lower()]
if key_dow:
    Wd = emb_layers[key_dow[0]].get_weights()[0]
    if Wd.shape[1] >= 2:
        dias = ['lun','mar','mie','jue','vie','sab','dom']
        plt.figure(figsize=(5,4))
        plt.scatter(Wd[:,0], Wd[:,1])
        for i,n in enumerate(dias[:Wd.shape[0]]):
            plt.annotate(n, (Wd[i,0], Wd[i,1]))
        plt.title('Embedding dia de la semana'); plt.grid(); plt.show()

---
## Comparativa final y reflexion

Tabla resumen de todas las estrategias probadas.

In [ ]:
tabla = pd.DataFrame([
    {'Estrategia': nombre, **{k:v for k,v in met.items() if k in ['R2_global','RMSPE_global']}}
    for nombre, met in resultados.items()
]).set_index('Estrategia').sort_values('R2_global', ascending=False)

print(tabla.to_string())
tabla

### Reflexion (rellenar con los resultados reales)

- **¿Que estrategia gana y por que?** *(rellenar)*
- **Diferencia entre el grupo que cierra domingos y el que no.** Las tiendas 1-5 son muy periodicas
  (ciclo semanal fuerte) -> R2 esperable mas alto. Las 562-769 son mas irregulares -> R2 menor.
- **Rol de los embeddings.** La Estrategia C aprende automaticamente que tiendas se parecen entre si.
  La visualizacion del embedding del dia de la semana muestra la separacion laborable/fin-de-semana.
- **Limitaciones.** El split (test = 2015 completo, >6 meses) es exigente; los ciclos anuales
  de navidad y verano solo aparecen una vez en train -> las estrategias con menos datos pueden
  no generalizar bien esos picos.
- **Lineas de mejora.** Modelo global entrenado con las 1.115 tiendas (Colab/GPU); seq2seq para
  horizonte multi-paso; Optuna para hiperparametros.